In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os
import pandas as pd
import numpy as np

# Get Data

In [2]:
kagglehub.login()


In [3]:
path = kagglehub.dataset_download("kazanova/sentiment140")


100%|██████████| 80.9M/80.9M [00:00<00:00, 110MB/s]

Extracting files...


In [4]:
csv_path = os.path.join(path, "training.1600000.processed.noemoticon.csv")

In [5]:
sent140 = pd.read_csv(csv_path, encoding="latin-1", header=None)

In [6]:
sent140.columns

Index([0, 1, 2, 3, 4, 5], dtype='int64')

In [7]:
path = kagglehub.dataset_download("snap/amazon-fine-food-reviews")

100%|██████████| 242M/242M [00:01<00:00, 138MB/s]

Extracting files...


In [8]:
csv_path=os.path.join(path,"Reviews.csv")

In [9]:
food=pd.read_csv(csv_path)

In [10]:
sent140.head()

,0,1,2,3,4,5
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [11]:
food.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


 Columns uniques

*   Score: 1 -> 5
*   HelpfulnessNumerator: 1 -> 808
*   HelpfulnessDenominator: 1 -> 815





# Encoding

## TFIDF

In [ ]:
!pip install rank_bm25

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi

In [13]:
food["Summary"] = food["Summary"].fillna("Not Available")

In [14]:
tfidf=TfidfVectorizer(stop_words='english',
                      max_features=5000,
                      min_df=5)

food_tfidf=tfidf.fit_transform(food['Summary'])

In [15]:
tfidf_df = pd.DataFrame(food_tfidf.toarray(), columns=tfidf.get_feature_names_out())

In [16]:
print("Feature names:", tfidf.get_feature_names_out())
print("Dense matrix shape:", food_tfidf.shape)
feature_names=tfidf.get_feature_names_out()

Feature names: ['00' '000' '10' ... 'zotz' 'zuke' 'zukes']
Dense matrix shape: (568454, 5000)


In [17]:
doc_index = 0

scores = food_tfidf[doc_index].toarray()[0]

word_scores = pd.Series(scores, index=feature_names)


In [18]:
top_words = word_scores.sort_values(ascending=False).head(100)

In [19]:
top_word_vectors=list(top_words.index[0:5])

In [20]:
tfidf_df[top_word_vectors]

,quality,food,dog,good,pizza
0,0.596823,0.506769,0.504101,0.364512,0.0
1,0.000000,0.000000,0.000000,0.000000,0.0
2,0.000000,0.000000,0.000000,0.000000,0.0
3,0.000000,0.000000,0.000000,0.000000,0.0
4,0.000000,0.000000,0.000000,0.000000,0.0
...,...,...,...,...,...
568449,0.000000,0.000000,0.000000,0.000000,0.0
568450,0.000000,0.000000,0.000000,0.000000,0.0
568451,0.000000,0.000000,0.000000,0.000000,0.0
568452,0.000000,0.000000,0.000000,0.000000,0.0


## BM25

In [21]:
food.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='object')

In [22]:
# bm25 with split
bm25 = BM25Okapi(food["Summary"].apply(lambda x: x.split(" ")))

In [23]:
query = "good"
tokenized_query = query.split(" ")
doc_scores = bm25.get_scores(tokenized_query)

In [24]:
good_num=bm25.get_top_n(tokenized_query, food["Summary"], n=10)

In [25]:
good_num

['good good',
 'good good',
 'good good',
 'good good',
 'good good',
 'good good',
 'good good',
 'good good',
 'good good',
 'good good soup!']

In [26]:
from sklearn.feature_extraction.text import CountVectorizer

In [27]:
df=food[:10_000]

In [28]:
tokenized_corpus = [doc.lower().split() for doc in df['Text']]

query = "excellent dark coffee"
query_tokens = query.lower().split()

In [31]:
count_vec = CountVectorizer()
count_matrix = count_vec.fit_transform(df['Text']).toarray()
vocab = count_vec.get_feature_names_out()

q_indices = [count_vec.vocabulary_[word] for word in query_tokens if word in count_vec.vocabulary_]
count_scores = np.sum(count_matrix[:, q_indices], axis=1)

tfidf_vec = TfidfVectorizer(norm='l2')
tfidf_matrix = tfidf_vec.fit_transform(df['Text']).toarray()
query_vec = tfidf_vec.transform([query]).toarray()[0]
tfidf_scores = np.dot(tfidf_matrix, query_vec)

bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)
bm25_scores = bm25.get_scores(query_tokens)

In [32]:
df['Count_Score'] = count_scores
df['TFIDF_Score'] = np.round(tfidf_scores, 4)
df['BM25_Score'] = np.round(bm25_scores, 4)

/tmp/ipykernel_10059/397437228.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Count_Score'] = count_scores
/tmp/ipykernel_10059/397437228.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['TFIDF_Score'] = np.round(tfidf_scores, 4)
/tmp/ipykernel_10059/397437228.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stabl

In [33]:
# Print a structured matrix comparing results
print(f"{'Review Snippet':<45} | {'Count':<5} | {'TF-IDF':<7} | {'BM25':<7}")
print("-" * 80)
for idx, row in df.iterrows():
    snippet = row['Text'] if len(row['Text']) < 43 else row['Text'][:40] + "..."
    print(f"{snippet:<45} | {row['Count_Score']:<5} | {row['TFIDF_Score']:<7} | {row['BM25_Score']:<7}")

Streaming output truncated to the last 5000 lines.
I thought I'd try the multi-pack to see ...   | 0     | 0.0     | 0.0    
i just started the paleo diet and i love...   | 0     | 0.0     | 0.0    
I've been Paleo for six months, as has m...   | 0     | 0.0     | 0.0    
I bought a 40 sampler pack from the cave...   | 0     | 0.0     | 0.0    
Formula itself is great. Works wonders f...   | 0     | 0.0     | 0.0    
With all of the choices of things to dri...   | 2     | 0.0907  | 4.0885 
This may not be the best tasting thing I...   | 2     | 0.0447  | 1.4049 
Let me preface this by saying I purchase...   | 0     | 0.0     | 0.0    
I love using generous amounts of vinegar...   | 0     | 0.0     | 0.0    
I bought this drink locally and I have t...   | 0     | 0.0     | 0.0    
This is the best ever.  Just add HOT wat...   | 0     | 0.0     | 0.0    
<a href="http://www.amazon.com/gp/produc...   | 1     | 0.067   | 4.0548 
We use these for our daughter for a few ...   | 0     | 0.0  

### Binning and Grouping by BM25 and TF-IDF Scores

In [34]:
bm25_bins = [-0.1, 0.5, 1.0, 1.5, 2.0, df['BM25_Score'].max() + 1]
bm25_labels = ['0-0.5', '0.5-1.0', '1.0-1.5', '1.5-2.0', '>2.0']
df['BM25_Score_Bin'] = pd.cut(df['BM25_Score'], bins=bm25_bins, labels=bm25_labels, right=False)

print("BM25 Score Bin Distribution:")
print(df.groupby('BM25_Score_Bin')['BM25_Score'].count())
print("\n")

tfidf_bins = [-0.1, 0.05, 0.1, 0.15, df['TFIDF_Score'].max() + 1]
tfidf_labels = ['0-0.05', '0.05-0.1', '0.1-0.15', '>0.15']
df['TFIDF_Score_Bin'] = pd.cut(df['TFIDF_Score'], bins=tfidf_bins, labels=tfidf_labels, right=False)

print("TF-IDF Score Bin Distribution:")
print(df.groupby('TFIDF_Score_Bin')['TFIDF_Score'].count())

BM25 Score Bin Distribution:
BM25_Score_Bin
0-0.5      8271
0.5-1.0      21
1.0-1.5      64
1.5-2.0     161
>2.0       1483
Name: BM25_Score, dtype: int64


TF-IDF Score Bin Distribution:
TFIDF_Score_Bin
0-0.05      8443
0.05-0.1     899
0.1-0.15     411
>0.15        247
Name: TFIDF_Score, dtype: int64


/tmp/ipykernel_10059/2945259929.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['BM25_Score_Bin'] = pd.cut(df['BM25_Score'], bins=bm25_bins, labels=bm25_labels, right=False)
/tmp/ipykernel_10059/2945259929.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('BM25_Score_Bin')['BM25_Score'].count())
/tmp/ipykernel_10059/2945259929.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.

### Binning and Grouping by Count Scores

In [35]:
max_count_score = df['Count_Score'].max()

# Example: 0, 1-2, 3-5, >5
if max_count_score < 1:
    count_bins = [-0.1, 0.5, max_count_score + 1]
    count_labels = ['0', '>0']
elif max_count_score < 5:
    count_bins = [x - 0.5 for x in range(int(max_count_score) + 2)]
    count_labels = [str(x) for x in range(int(max_count_score) + 1)]
else:
    # Custom bins for larger ranges
    count_bins = [-0.1, 0.5, 2.5, 5.5, max_count_score + 1]
    count_labels = ['0', '1-2', '3-5', '>5']

df['Count_Score_Bin'] = pd.cut(df['Count_Score'], bins=count_bins, labels=count_labels, right=True)

print("Count Score Bin Distribution:")
print(df.groupby('Count_Score_Bin')['Count_Score'].count())

Count Score Bin Distribution:
Count_Score_Bin
0      7857
1-2    1569
3-5     461
>5      113
Name: Count_Score, dtype: int64


/tmp/ipykernel_10059/1850792071.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Count_Score_Bin'] = pd.cut(df['Count_Score'], bins=count_bins, labels=count_labels, right=True)
/tmp/ipykernel_10059/1850792071.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('Count_Score_Bin')['Count_Score'].count())


In [41]:
from sklearn.metrics.pairwise import cosine_similarity
words_to_test = ["delicious", "tasty", "spoiled", "coffee"]

tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(words_to_test).toarray()
tfidf_vocab = {word: tfidf_matrix[i] for i, word in enumerate(words_to_test)}

## Comparison between TFIDF and Word Embeddings

In [44]:
pairs = [("delicious", "tasty"), ("delicious", "spoiled")]

print(f"{'Word Pair':<25} | {'TF-IDF Similarity':<20}")
print("-" * 50)

for w1, w2 in pairs:
    v1_t, v2_t = tfidf_vocab[w1].reshape(1, -1), tfidf_vocab[w2].reshape(1, -1)
    sim_tfidf = cosine_similarity(v1_t, v2_t)[0][0]

    print(f"{w1 + ' <-> ' + w2:<25} | {sim_tfidf:<20.4f}")

Word Pair                 | TF-IDF Similarity   
--------------------------------------------------
delicious <-> tasty       | 0.0000              
delicious <-> spoiled     | 0.0000              


In [48]:
!pip install gensim
import gensim.downloader as api
print("Load glove")
dense_model = api.load("glove-wiki-gigaword-50")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 32.7 MB/s eta 0:00:00
Load glove
[==================================================] 100.0% 66.0/66.0MB downloaded


In [60]:
vocabulary = list(set([word for pair in word_pairs for word in pair]))
tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(vocabulary).toarray()

tfidf_lookup = {word: tfidf_matrix[i] for i, word in enumerate(vocabulary)}

word_pairs = [
    ("delicious", "tasty"),
    ("coffee", "espresso"),
    ("delicious", "spoiled"),
    ("coffee", "software")
]

comparison_data = []

for w1, w2 in word_pairs:
    v1_tfidf = tfidf_lookup[w1].reshape(1, -1)
    v2_tfidf = tfidf_lookup[w2].reshape(1, -1)
    similarity_tfidf = cosine_similarity(v1_tfidf, v2_tfidf)[0][0]

    similarity_dense = dense_model.similarity(w1, w2)

    comparison_data.append({
        "Word 1": w1,
        "Word 2": w2,
        "TF-IDF Similarity": round(similarity_tfidf, 4),
        "Dense Embedding Similarity": round(similarity_dense, 4)
    })

# 3. Print the Structured Output Matrix
df_results = pd.DataFrame(comparison_data)
print("\n" + "="*70 + "\nMATHEMATICAL SEMANTIC SIMILARITY MATRIX\n" + "="*70)
print(df_results.to_string(index=False))


MATHEMATICAL SEMANTIC SIMILARITY MATRIX
   Word 1   Word 2  TF-IDF Similarity  Dense Embedding Similarity
delicious    tasty                0.0                      0.9297
   coffee espresso                0.0                      0.6471
delicious  spoiled                0.0                      0.5500
   coffee software                0.0                      0.3667


Dense Embedding show that here is semantic meaning behind each pair of word, TFIDF does not.